# Construção e auditoria do corpus de Machado de Assis

**Projeto:** RNAP 2026/2 — modelo de linguagem autorregressivo  
**Objetivo deste notebook:** documentar a escolha da fonte, inventariar os textos, registrar as decisões de inclusão, construir um arquivo mestre UTF-8 e produzir relatórios de auditoria reproduzíveis.

## Decisão sobre as fontes

A base escolhida é o pacote `machado` distribuído por **NLTK Data**. Seu manifesto identifica o pacote como *Machado de Assis — Obra Completa*, declara domínio público, aponta como página de origem o portal [Machado de Assis — Vida e Obra, MEC](https://machado.mec.gov.br/) e publica SHA-256. Esse caminho é reproduzível e tem proveniência mais clara que baixar uma cópia sem versão pelo Kaggle. NLTK é o distribuidor do pacote; o MEC é a fonte bibliográfica indicada pelo próprio pacote.

Fontes usadas para controle:

- [Bibliografia da Academia Brasileira de Letras (ABL)](https://www.academia.org.br/academicos/machado-de-assis/bibliografia): lista de obras em formato de livro para o cruzamento bibliográfico; não é tratada como inventário absolutamente exaustivo de toda a produção.
- [Corpus Machado no NLTK Data](https://github.com/nltk/nltk_data/tree/gh-pages/packages/corpora) e [manifesto do pacote](https://github.com/nltk/nltk_data/blob/gh-pages/packages/corpora/machado.xml).
- [Catálogo de obras digitalizadas do MEC](https://machado.mec.gov.br/obra-completa-lista).
- [Dataset Kaggle de luxedo](https://www.kaggle.com/datasets/luxedo/machado-de-assis): usado como referência secundária, não incorporado automaticamente. Um [catálogo secundário de dados abertos](https://cienciaaberta.org/dados-abertos/outros-dados-linguisticos/) descreve 116 itens em categorias como crítica, crônica e tradução. Não consegui confirmar a composição do snapshot diretamente no Kaggle; a equivalência com os arquivos locais não foi verificada e semelhança na contagem não prova equivalência.

## O que “completo” significa aqui

O arquivo-fonte do NLTK contém **246 arquivos .txt**, enquanto o índice interno lista **116 itens**. A diferença decorre de 130 arquivos de contos avulsos além dos sete arquivos de coletâneas. A inclusão automática dos 246 criaria risco de reapresentar contos também presentes nas coletâneas. Por isso:

1. A base indexada tem 116 itens. Exclui os três textos da categoria `traducao` e *Queda que as mulheres têm para os tolos* (tradução dentro de miscelânea), restando **112 itens autorais indexados**.
2. Os 130 contos avulsos são comparados aos sete volumes de contos. Os que não mostram sobreposição alta entram como textos autorais adicionais, em seções próprias e com proveniência individual; os demais ficam pendentes.
3. Textos críticos, crônicas e miscelânea são escritos originais de Machado presentes no índice; são identificados por categoria, embora nem todos sejam livros autônomos.
4. Nem 112 nem a contagem final de itens devem ser reportadas como quantidade de livros. O cruzamento com ABL mostra livros, conteúdos distribuídos por várias peças e lacunas editoriais. Este é um **corpus autoral expandido baseado na coleção digital do MEC/NLTK**, não uma alegação de que a obra integral definitiva de Machado foi esgotada.

O pacote declara domínio público, mas seus próprios textos citam transcrições de edições modernas (por exemplo, Nova Aguilar, 1994, e em alguns contos uma edição de 2008). Preservaremos essa proveniência e não apagaremos notas editoriais às cegas. Para uso acadêmico do texto autoral, a equipe deve confirmar as condições aplicáveis à transcrição/edição escolhida e citar a fonte corretamente.

## Artefatos gerados

Todos os produtos são gerados por este notebook na pasta `dados/`: ZIP original, textos-fonte extraídos, inventário CSV, auditoria CSV, cruzamento com a ABL, corpus mestre UTF-8 e manifesto com hashes. O texto mestre preserva as transcrições-fonte sem normalização destrutiva e usa delimitadores de obra adicionados pelo projeto.

## 1. Configuração e proveniência fixada

O SHA-256 publicado no índice NLTK Data é usado como verificação. O ZIP já foi baixado para esta pasta durante a pesquisa; se o notebook for executado em outro ambiente, a célula seguinte o baixa novamente quando necessário. A extração aceita somente caminhos internos seguros do ZIP.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import hashlib
import json
import re
import csv
import unicodedata
from collections import Counter
from datetime import date

PROJECT_DIR = Path.cwd()
# Permite executar tanto na raiz do workspace quanto a partir de projetos/machado-assis.
if not (PROJECT_DIR / "projetos" / "machado-assis").exists():
    if PROJECT_DIR.name == "machado-assis":
        PROJECT_DIR = PROJECT_DIR.parent.parent
    else:
        raise FileNotFoundError("Execute este notebook da raiz do workspace ou da pasta projetos/machado-assis.")

MACHADO_DIR = PROJECT_DIR / "projetos" / "machado-assis"
DATA_DIR = MACHADO_DIR / "dados"
RAW_DIR = DATA_DIR / "raw" / "machado"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_URL = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/machado.zip"
ARCHIVE_SHA256 = "772463b1553c1b0ff1fc0360768b31f59b488f7a52d44cc92c3e31ca289acce9"
ARCHIVE_PATH = DATA_DIR / "machado-nltk.zip"
SOURCE_PAGE = "https://machado.mec.gov.br/"
NLTK_MANIFEST = "https://github.com/nltk/nltk_data/blob/gh-pages/packages/corpora/machado.xml"
ABL_BIBLIOGRAPHY = "https://www.academia.org.br/academicos/machado-de-assis/bibliografia"

print("Pasta do projeto:", MACHADO_DIR)
print("Pacote-fonte:", ARCHIVE_PATH)
print("SHA-256 esperado:", ARCHIVE_SHA256)

Pasta do projeto: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis
Pacote-fonte: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/machado-nltk.zip
SHA-256 esperado: 772463b1553c1b0ff1fc0360768b31f59b488f7a52d44cc92c3e31ca289acce9


In [2]:
if not ARCHIVE_PATH.exists():
    print("Baixando pacote-fonte do NLTK Data...")
    urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)

archive_hash = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
print("SHA-256 calculado:", archive_hash)
assert archive_hash == ARCHIVE_SHA256, (
    "O arquivo não corresponde ao snapshot identificado pelo manifesto NLTK. "
    "Não extraia nem use este arquivo; verifique a versão da fonte."
)
print("Integridade do pacote: OK")

SHA-256 calculado: 772463b1553c1b0ff1fc0360768b31f59b488f7a52d44cc92c3e31ca289acce9
Integridade do pacote: OK


## 2. Extração segura e inventário do arquivo-fonte

Os arquivos de texto do ZIP foram verificados como Windows-1252 (CP1252), não UTF-8. O ZIP original é preservado sem alterações; as cópias extraídas são convertidas corretamente para UTF-8, incluindo travessões e aspas tipográficas.

In [3]:
with ZipFile(ARCHIVE_PATH) as zf:
    members = zf.namelist()
    unsafe = []
    for name in members:
        target = (DATA_DIR / "raw" / name).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            unsafe.append(name)
    assert not unsafe, f"Há caminhos inseguros no arquivo: {unsafe[:5]}"
    zf.extractall(DATA_DIR / "raw")
    # Converte cópias de trabalho; o ZIP permanece como fonte byte a byte.
    for name in members:
        if name.startswith("machado/") and not name.endswith("/"):
            basename = Path(name).name
            if name.lower().endswith(".txt") or basename in {"README", "CONTENTS"}:
                relative = Path(name).relative_to("machado")
                target = RAW_DIR / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(zf.read(name).decode("cp1252"), encoding="utf-8", newline="\n")

SOURCE_ROOT = RAW_DIR
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Esperava os textos extraídos em {SOURCE_ROOT}")

source_files = sorted(SOURCE_ROOT.rglob("*.txt"))
print("Membros do ZIP:", len(members))
print("Arquivos .txt:", len(source_files))
print("Categorias:", sorted({p.parent.name for p in source_files}))
print("Distribuição:", dict(sorted(Counter(p.parent.name for p in source_files).items())))

Membros do ZIP: 257
Arquivos .txt: 246
Categorias: ['contos', 'critica', 'cronica', 'miscelanea', 'poesia', 'romance', 'teatro', 'traducao']
Distribuição: {'contos': 137, 'critica': 45, 'cronica': 24, 'miscelanea': 10, 'poesia': 7, 'romance': 10, 'teatro': 10, 'traducao': 3}


In [4]:
contents_path = SOURCE_ROOT / "CONTENTS"
if not contents_path.exists():
    raise FileNotFoundError("Arquivo CONTENTS ausente no pacote-fonte.")

contents_text = contents_path.read_text(encoding="utf-8")
catalog_rows = []
for line in contents_text.splitlines():
    match = re.match(r"^([\w-]+/[^:\n]+\.txt):\s*(.*?)\s*$", line)
    if match:
        relpath, title = match.groups()
        catalog_rows.append({"source_file": relpath, "title_in_contents": title})

catalog_df = {row["source_file"]: row for row in catalog_rows}
print("Itens enumerados em CONTENTS:", len(catalog_rows))
print("Arquivos .txt enumerados:", len(catalog_df))
print("Itens por categoria no índice:", dict(sorted(Counter(x.split("/")[0] for x in catalog_df).items())))
assert len(catalog_rows) == 116, "A contagem mudou; revise o índice antes de continuar."
assert len(catalog_df) == len(catalog_rows), "Há caminhos duplicados no índice."

Itens enumerados em CONTENTS: 116
Arquivos .txt enumerados: 116
Itens por categoria no índice: {'contos': 7, 'critica': 45, 'cronica': 24, 'miscelanea': 10, 'poesia': 7, 'romance': 10, 'teatro': 10, 'traducao': 3}


## 3. Regras de seleção para o corpus mestre

A inclusão inicial usa o índice do próprio pacote, não uma inferência por nome de arquivo. Excluímos todo o diretório `traducao` e a obra traduzida *Queda que as mulheres têm para os tolos* (1861), dentro de `miscelanea`. Os 130 contos não enumerados em `CONTENTS` entram após a comparação textual da seção 8 somente quando não há sobreposição alta com as sete coletâneas.

Esta decisão cria uma versão principal conservadora e reproduzível. Ela não resolve obras que ABL lista como coletâneas póstumas nem demonstra que toda peça dispersa na imprensa está presente.

In [5]:
# Controle bibliográfico: títulos listados pela ABL e correspondência com o pacote.
# "Distribuído/parte" significa que o texto pode estar representado por peças ou volumes,
# não que a identidade integral da edição da ABL tenha sido comprovada.
abl_crosswalk = [
("Queda que as mulheres têm para os tolos", "miscelanea/mams02.txt", "excluído: tradução"),
("Desencantos", "teatro/matt03.txt", "localizado"),
("Teatro", "teatro/*.txt", "representado por peças; sem volume único"),
("Quase ministro", "teatro/matt05.txt", "localizado"),
("Crisálidas", "poesia/maps01.txt", "localizado"),
("Os deuses de casaca", "teatro/matt06.txt", "localizado"),
("Falenas", "poesia/maps02.txt", "localizado"),
("Contos fluminenses", "contos/macn001.txt", "localizado como coletânea"),
("Ressurreição", "romance/marm01.txt", "localizado"),
("Histórias da meia-noite", "contos/macn002.txt", "localizado como coletânea"),
("A mão e a luva", "romance/marm02.txt", "localizado"),
("Americanas", "poesia/maps03.txt", "localizado"),
("Helena", "romance/marm03.txt", "localizado"),
("Iaiá Garcia", "romance/marm04.txt", "localizado"),
("Memórias póstumas de Brás Cubas", "romance/marm05.txt", "localizado"),
("Tu, só tu, puro amor", "teatro/matt08.txt", "localizado; ABL data 1881, NLTK 1880"),
("Papéis avulsos", "contos/macn003.txt", "localizado como coletânea"),
("Histórias sem data", "contos/macn004.txt", "localizado como coletânea"),
("Quincas Borba", "romance/marm07.txt", "localizado"),
("Várias histórias", "contos/macn005.txt", "localizado como coletânea"),
("Páginas recolhidas", "contos/macn006.txt", "localizado como coletânea"),
("Dom Casmurro", "romance/marm08.txt", "localizado"),
("Poesias completas", "poesia/*.txt", "conteúdos poéticos distribuídos; edição integral não verificada"),
("Esaú e Jacó", "romance/marm09.txt", "localizado"),
("Relíquias de casa velha", "contos/macn007.txt", "localizado como coletânea"),
("Memorial de Aires", "romance/marm10.txt", "localizado"),
("Crítica (1910)", "critica/*.txt", "peças críticas; edição integral não verificada"),
("Outras relíquias (1910)", "miscelanea/*.txt; critica/*.txt", "correspondência não estabelecida"),
("Correspondência (1932)", "nenhum arquivo identificado", "lacuna"),
("Crônicas, 4 volumes (1937)", "cronica/*.txt", "séries de crônicas; equivalência integral não verificada"),
("Crítica literária (1937)", "critica/*.txt", "peças críticas; edição integral não verificada"),
("Casa velha (1944)", "romance/marm06.txt", "localizado"),
]

def include_indexed(relpath):
    category = relpath.split("/", 1)[0]
    if category == "traducao":
        return False, "tradução integral de obra alheia"
    if relpath == "miscelanea/mams02.txt":
        return False, "tradução: Queda que as mulheres têm para os tolos"
    return True, "item autoral indexado pelo pacote NLTK/MEC"

abl_path = DATA_DIR / "cruzamento_bibliografia_abl.csv"
with abl_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["obra_ABL", "referencia_no_pacote", "situacao"])
    writer.writerows(abl_crosswalk)
print("Itens no cruzamento ABL:", len(abl_crosswalk))
print("Cruzamento salvo em:", abl_path)

Itens no cruzamento ABL: 32
Cruzamento salvo em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/cruzamento_bibliografia_abl.csv


In [6]:
all_relpaths = sorted(
    p.relative_to(SOURCE_ROOT).as_posix()
    for p in SOURCE_ROOT.rglob("*.txt")
)
inventory = []
for relpath in all_relpaths:
    category = relpath.split("/", 1)[0]
    indexed = relpath in catalog_df
    if indexed:
        title = catalog_df[relpath]["title_in_contents"]
        include, reason = include_indexed(relpath)
        role = "item_indexado"
    else:
        include = False
        reason = "conto avulso fora do índice de 116 itens; requer análise de sobreposição/escopo"
        role = "conto_avulso" if category == "contos" else "arquivo_fora_do_indice"
        # O cabeçalho do NLTK contém título e ano. É apenas candidato para conferência humana.
        try:
            first_line = (SOURCE_ROOT / relpath).read_text(encoding="utf-8").splitlines()[0].strip()
        except Exception:
            first_line = ""
        title = first_line
    inventory.append({
        "source_file": relpath,
        "category": category,
        "title_in_contents": title,
        "in_index_116": indexed,
        "role": role,
        "include_in_master": include,
        "decision": reason,
    })

inventory_path = DATA_DIR / "inventario_fontes.csv"
with inventory_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(inventory[0].keys()))
    writer.writeheader()
    writer.writerows(inventory)

print("Total de arquivos inventariados:", len(inventory))
print("Itens indexados incluídos:", sum(x["include_in_master"] for x in inventory))
print("Itens de tradução excluídos:", sum(x["role"] == "item_indexado" and not x["include_in_master"] for x in inventory))
print("Contos avulsos preservados para auditoria:", sum(x["role"] == "conto_avulso" for x in inventory))
print("Inventário salvo em:", inventory_path)
assert len(inventory) == 246

Total de arquivos inventariados: 246
Itens indexados incluídos: 112
Itens de tradução excluídos: 4
Contos avulsos preservados para auditoria: 130
Inventário salvo em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/inventario_fontes.csv


In [7]:
# Auditoria por arquivo: preserva bytes-fonte, decodificação, tamanhos e hashes.
audit = []
for row in inventory:
    path = SOURCE_ROOT / row["source_file"]
    with ZipFile(ARCHIVE_PATH) as original_zip:
        raw_bytes = original_zip.read("machado/" + row["source_file"])
    text = path.read_text(encoding="utf-8")
    audit.append({
        **row,
        "bytes_source_archive": len(raw_bytes),
        "caracteres_decodificados": len(text),
        "palavras_aprox": len(text.split()),
        "sha256_fonte": hashlib.sha256(raw_bytes).hexdigest(),
        "mojibake_patterns": sum(text.count(x) for x in ("Ã£", "Ã¡", "Ã©", "Ã­", "Ã³", "Ãº", "Ã§", "Ãµ", "Ã¢", "Ãª", "Ã´", "â€™", "â€œ", "â€")),
        "controles_C1": sum("\u0080" <= ch <= "\u009f" for ch in text),
        "tem_replacement_character": "\\ufffd" in text,
        "tem_marca_edicao_moderna": any(x in text[:1500].lower() for x in ["nova aguilar", "martins fontes", "edições loyola", "edições w. m. jackson"]),
    })

audit_path = DATA_DIR / "auditoria_arquivos.csv"
with audit_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(audit[0].keys()))
    writer.writeheader()
    writer.writerows(audit)

included = [x for x in audit if x["include_in_master"]]
print("Textos no mestre:", len(included))
print("Caracteres no mestre (antes de delimitadores):", sum(x["caracteres_decodificados"] for x in included))
print("Arquivos com caractere U+FFFD:", sum(x["tem_replacement_character"] for x in audit))
print("Ocorrências de padrões comuns de mojibake:", sum(x["mojibake_patterns"] for x in audit))
print("Controles Unicode C1 nas cópias UTF-8:", sum(x["controles_C1"] for x in audit))
print("Arquivos citando edições modernas no cabeçalho:", sum(x["tem_marca_edicao_moderna"] for x in audit))
print("Auditoria salva em:", audit_path)

Textos no mestre: 112
Caracteres no mestre (antes de delimitadores): 10099374
Arquivos com caractere U+FFFD: 0
Ocorrências de padrões comuns de mojibake: 0
Controles Unicode C1 nas cópias UTF-8: 0
Arquivos citando edições modernas no cabeçalho: 142
Auditoria salva em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/auditoria_arquivos.csv


## 4. Duplicação e arquivos de contos avulsos

A auditoria usa dois sinais sem apagar nenhum arquivo-fonte: SHA-256 dos bytes e hash de uma representação comparável que remove espaços, diferenças de caixa e diacríticos. Hashes iguais identificam cópias exatas/normalizadas, mas hashes diferentes **não** provam que os textos sejam distintos. Contos avulsos precisam de comparação textual mais cuidadosa com as coletâneas para futura ampliação do corpus.

In [8]:
def comparison_fingerprint(text):
    # Chave apenas para localizar cópias prováveis; nunca substitui nem modifica o texto-fonte.
    normalized = unicodedata.normalize("NFKD", text.casefold())
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    normalized = "".join(ch for ch in normalized if ch.isalnum())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

fingerprints = {}
for row in audit:
    text = (SOURCE_ROOT / row["source_file"]).read_text(encoding="utf-8")
    fingerprints.setdefault(comparison_fingerprint(text), []).append(row["source_file"])

duplicate_groups = [paths for paths in fingerprints.values() if len(paths) > 1]
print("Grupos com texto integral igual após normalização conservadora:", len(duplicate_groups))
for group in duplicate_groups[:20]:
    print(" •", " | ".join(group))
print("Contos avulsos fora do mestre:", sum(x["role"] == "conto_avulso" for x in inventory))
print("Ação:", "nenhum arquivo-fonte apagado; a ampliação depende de revisão das sobreposições.")

Grupos com texto integral igual após normalização conservadora: 0
Contos avulsos fora do mestre: 130
Ação: nenhum arquivo-fonte apagado; a ampliação depende de revisão das sobreposições.


## 5. Construção do arquivo mestre

A versão inicial contém 112 itens indexados considerados autorais. Após a seção 8, contos avulsos sem sobreposição alta serão acrescentados como textos separados. Cada texto é decodificado de Windows-1252 para Unicode e as cópias de trabalho são gravadas em UTF-8. O conteúdo é preservado exatamente após a decodificação, inclusive cabeçalhos, índices e notas de fonte presentes no pacote; delimitadores de obra são adicionados para rastreabilidade. Como notas editoriais não foram removidas automaticamente, antes do treinamento será possível criar uma versão de modelagem revisada sem perder esta fonte mestre.

In [9]:
MASTER_PATH = DATA_DIR / "corpus_machado_autoral_nltk_mec.txt"
parts = []
for row in included:
    text = (SOURCE_ROOT / row["source_file"]).read_text(encoding="utf-8")
    label = row["title_in_contents"].replace("\\n", " ").strip()
    parts.append(
        f"\n\n===== INÍCIO DO ITEM: {label} | categoria={row['category']} | fonte={row['source_file']} =====\n\n"
        + text
        + f"\n\n===== FIM DO ITEM: {row['source_file']} =====\n"
    )
master_text = "".join(parts)
MASTER_PATH.write_text(master_text, encoding="utf-8", newline="\n")

master_hash = hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest()
print("Mestre UTF-8:", MASTER_PATH)
print("Itens:", len(included))
print("Caracteres:", len(master_text))
print("Palavras aproximadas:", len(master_text.split()))
print("Tamanho (bytes UTF-8):", MASTER_PATH.stat().st_size)
print("SHA-256:", master_hash)
print("\nAmostra inicial (somente inspeção):\n")
print(master_text[:1200])

Mestre UTF-8: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/corpus_machado_autoral_nltk_mec.txt
Itens: 112
Caracteres: 10118130


Palavras aproximadas: 1731690
Tamanho (bytes UTF-8): 10446484
SHA-256: 75713d2864f1ad85a26e2168866387eaca34a03ddcb03e17952c7bd185f6fb0d

Amostra inicial (somente inspeção):



===== INÍCIO DO ITEM: Contos Fluminenses (1870); Miss Dollar; Luís Soares; A mulher de preto; O segredo de Augusta; Confissões de uma viúva moça; Linha reta e linha curva; Frei Sim | categoria=contos | fonte=contos/macn001.txt =====

Conto, Contos Fluminenses, 1870

Contos Fluminenses

Texto-fonte:

Obra Completa, Machado de Assis, vol. II,

Rio de Janeiro: Nova Aguilar, 1994.

Publicado originalmente pela
Editora Garnier, Rio de Janeiro, em 1870.

ÍNDICE

MISS DOLLAR

LUÍS
SOARES

A MULHER DE
PRETO

O
SEGREDO DE AUGUSTA

CONFISSÕES DE UMA VIÚVA MOÇA

LINHA
RETA E LINHA CURVA

FREI
SIMÃO

MISS
DOLLAR

ÍNDICE

Capítulo Primeiro

Capítulo II

Capítulo iii

Capítulo iv

Capítulo v

Capítulo vI

Capítulo vII

CAPÍTULO VIII

CAPÍTULO PRIMEIRO

Era conveniente ao romance que o leitor
ficasse muito tempo sem saber quem 

## 6. Verificações de integridade e manifesto

Os testes abaixo confirmam consistência estrutural, não completude filológica. “Passou” significa que os arquivos e decisões documentadas correspondem ao snapshot baixado. As lacunas do cruzamento ABL permanecem visíveis para análise humana.

In [10]:
assert archive_hash == ARCHIVE_SHA256
assert len(source_files) == 246
assert len(catalog_rows) == 116
assert len(included) == 112
assert len([x for x in audit if x["role"] == "conto_avulso"]) == 130
assert MASTER_PATH.exists() and MASTER_PATH.stat().st_size > 0
assert MASTER_PATH.read_bytes().decode("utf-8") == master_text
assert all((SOURCE_ROOT / x["source_file"]).exists() for x in included)
assert len({x["source_file"] for x in included}) == len(included)
assert not any(x["source_file"].startswith("traducao/") for x in included)
assert not any(x["source_file"] == "miscelanea/mams02.txt" for x in included)
print("Integridade estrutural: PASSOU")
print("Completude de toda a produção de Machado: NÃO AFIRMADA")
print("Revisão pendente: comparar itens ABL marcados como distribuídos/lacunas e decidir tratamento editorial.")

Integridade estrutural: PASSOU
Completude de toda a produção de Machado: NÃO AFIRMADA
Revisão pendente: comparar itens ABL marcados como distribuídos/lacunas e decidir tratamento editorial.


In [11]:
manifest = {
    "project": "RNAP 2026/2 — corpus Machado de Assis",
    "created_utc_date": date.today().isoformat(),
    "source_name": "NLTK Data package machado",
    "source_download_url": ARCHIVE_URL,
    "upstream_declared_work": "Machado de Assis — Obra Completa",
    "upstream_declared_license": "Public Domain",
    "upstream_origin_page": SOURCE_PAGE,
    "upstream_manifest": NLTK_MANIFEST,
    "archive_sha256_expected": ARCHIVE_SHA256,
    "archive_sha256_calculated": archive_hash,
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "source_text_encoding": "Windows-1252 (CP1252), verificado nos membros do ZIP",
    "extracted_text_encoding": "UTF-8",
    "extracted_text_directory": "raw/machado/",
    "archive_txt_files": len(inventory),
    "indexed_items": len(catalog_rows),
    "included_authorial_indexed_items": len(included),
    "excluded_translations": [
        "traducao/matr01.txt",
        "traducao/matr02.txt",
        "traducao/matr03.txt",
        "miscelanea/mams02.txt (Queda que as mulheres têm para os tolos)",
    ],
    "unindexed_short_story_files_preserved_for_review": sum(x["role"] == "conto_avulso" for x in inventory),
    "master_file": MASTER_PATH.name,
    "master_sha256": master_hash,
    "master_utf8_bytes": MASTER_PATH.stat().st_size,
    "control_bibliography": ABL_BIBLIOGRAPHY,
    "limitations": [
        "112 itens não equivalem a 112 livros.",
        "A coleção não foi provada exaustiva contra toda a produção bibliográfica.",
        "130 contos avulsos não foram concatenados até revisão de sobreposição/escopo.",
        "Arquivos-fonte citam transcrições de edições modernas; preservar e revisar a proveniência editorial.",
        "O corpus mestre mantém notas editoriais e cabeçalhos da transcrição.",
    ],
}
manifest_path = DATA_DIR / "manifesto_corpus.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Manifesto salvo:", manifest_path)
print("Corpus mestre salvo:", MASTER_PATH)
print("ZIP original preservado em:", ARCHIVE_PATH)
print("Cópias UTF-8 extraídas em:", SOURCE_ROOT)

Manifesto salvo: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/manifesto_corpus.json
Corpus mestre salvo: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/corpus_machado_autoral_nltk_mec.txt
ZIP original preservado em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/machado-nltk.zip
Cópias UTF-8 extraídas em: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/raw/machado


## 7. Decisões para registrar no relatório do projeto

Descreva a coleção como itens autorais indexados do pacote NLTK Data associado ao portal do MEC, mais os contos avulsos sem sobreposição alta com as coletâneas, montados em UTF-8. Não chame itens de livros nem afirme que a coleção cobre toda a bibliografia: o cruzamento da ABL ainda marca conteúdos distribuídos e uma lacuna.

No experimento, reserve obras inteiras para avaliação antes do treinamento do modelo; uma divisão aleatória de trechos do mesmo livro pode produzir vazamento de estilo e conteúdo. Se futuramente entrar uma obra de avaliação fora do arquivo de treino, mantenha o corpus mestre como acervo e gere um arquivo de treino derivado, excluindo explicitamente essa obra. Qualquer limpeza editorial deve ser aplicada numa nova versão, com regras e hashes documentados, nunca sobrescrevendo `raw/`.

**Próxima revisão filológica:** comparar uma amostra e depois as lacunas do cruzamento com os fac-símiles das primeiras edições (Biblioteca Brasiliana/USP e Biblioteca Nacional), conferir as obras póstumas/compilações e decidir se os contos avulsos não incluídos representam textos inéditos em livro. Isso permite qualificar a cobertura sem confundir quantidade de arquivos com completude.

## 8. Auditoria textual dos 130 contos avulsos

A comparação abaixo tira dos contos avulsos o cabeçalho bibliográfico anterior a “Publicado originalmente” e procura sobreposição textual com cada uma das sete coletâneas do corpus mestre. A chave remove diferenças de caixa, acentos, pontuação e espaços **somente para comparação**. O conteúdo original não é alterado.

A medida é a fração de sequências de 32 caracteres distintas do conto que também aparecem na coletânea. Pontuação:
- **contido**: o texto normalizado do corpo aparece por inteiro em um volume;
- **sobreposição alta (≥ 0,85)**: forte evidência de que a maior parte do conto está num volume, mas variação editorial impede declarar identidade;
- **sobreposição parcial (≥ 0,40)**: há trecho comum; exige leitura;
- **sem sobreposição alta (< 0,40)**: não localizado por este método, não prova ineditismo nem ausência em outra edição;
- **não comparável**: não foi possível isolar o corpo com a regra bibliográfica.

Este é um filtro de triagem reproduzível, não uma equivalência filológica. Casos parciais ou não comparáveis permanecem pendentes. Casos sem sobreposição alta são acrescentados como textos autorais avulsos; isso não prova que nunca tenham aparecido em outra coletânea ou edição.

In [12]:
def normalize_for_match(text):
    text = unicodedata.normalize("NFKD", text.casefold())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return "".join(ch for ch in text if ch.isalnum())

def story_body(text):
    # A nota às vezes quebra a expressão em linhas separadas: “Publicado\noriginalmente”.
    marker = text.casefold().find("publicado")
    if marker < 0:
        return None
    remainder = text[marker:]
    if remainder[:120].casefold().split()[:2] != ["publicado", "originalmente"]:
        return None
    boundary = remainder.find("\n\n")
    if boundary < 0:
        return None
    body = remainder[boundary:].strip()
    return body if len(body) >= 200 else None

collection_rows = [
    x for x in inventory
    if x["source_file"] in {f"contos/macn{i:03d}.txt" for i in range(1, 8)}
]
assert len(collection_rows) == 7
collection_texts = {
    x["source_file"]: normalize_for_match(
        (SOURCE_ROOT / x["source_file"]).read_text(encoding="utf-8")
    )
    for x in collection_rows
}
NGRAM_SIZE = 32
collection_ngrams = {
    path: {text[i:i+NGRAM_SIZE] for i in range(max(0, len(text)-NGRAM_SIZE+1))}
    for path, text in collection_texts.items()
}
all_collection_text = "\\n".join(collection_texts.values())
all_collection_ngrams = set().union(*collection_ngrams.values())

loose_stories = sorted(
    x["source_file"] for x in inventory if x["role"] == "conto_avulso"
)
story_audit = []
for relpath in loose_stories:
    raw = (SOURCE_ROOT / relpath).read_text(encoding="utf-8")
    body = story_body(raw)
    if body is None:
        story_audit.append({
            "source_file": relpath,
            "title_candidate": raw.splitlines()[0].strip(),
            "body_chars": "",
            "best_collection": "",
            "best_overlap": "",
            "whole_body_contained": False,
            "decision": "não comparável: revisar cabeçalho/manual",
        })
        continue

    normalized = normalize_for_match(body)
    grams = {normalized[i:i+NGRAM_SIZE] for i in range(max(0, len(normalized)-NGRAM_SIZE+1))}
    if not grams:
        best_path, best_score, contained = "", 0.0, False
    else:
        scored = [
            (len(grams & volume_grams) / len(grams), path)
            for path, volume_grams in collection_ngrams.items()
        ]
        best_score, best_path = max(scored)
        contained = any(normalized in volume for volume in collection_texts.values())

    if contained:
        decision = "contido: corpo integral localizado após normalização"
    elif best_score >= 0.85:
        decision = "sobreposição alta: provável repetição com variante textual"
    elif best_score >= 0.40:
        decision = "sobreposição parcial: requer revisão humana"
    else:
        decision = "sem sobreposição alta: possível texto adicional; requer revisão de escopo"
    story_audit.append({
        "source_file": relpath,
        "title_candidate": raw.splitlines()[0].strip(),
        "body_chars": len(body),
        "best_collection": best_path,
        "best_overlap": round(best_score, 4),
        "whole_body_contained": contained,
        "decision": decision,
    })

story_audit_path = DATA_DIR / "auditoria_contos_avulsos.csv"
with story_audit_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(story_audit[0].keys()))
    writer.writeheader()
    writer.writerows(story_audit)

story_counts = Counter(x["decision"].split(":")[0] for x in story_audit)
print("Contos avulsos comparados:", len(story_audit))
print("Resultados:", dict(story_counts))
print("Relatório detalhado:", story_audit_path)
print("\\nCasos a revisar:")
for row in story_audit:
    if row["decision"].startswith(("sobreposição parcial", "sem sobreposição", "não comparável")):
        print(row["source_file"], row["title_candidate"], row["best_collection"], row["best_overlap"], row["decision"])

Contos avulsos comparados: 130
Resultados: {'sem sobreposição alta': 130}
Relatório detalhado: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/auditoria_contos_avulsos.csv
\nCasos a revisar:
contos/macn008.txt Conto, Três Tesouros Perdidos, 1858 contos/macn007.txt 0.0 sem sobreposição alta: possível texto adicional; requer revisão de escopo
contos/macn009.txt Conto, O País das quimeras, 1862 contos/macn007.txt 0.0 sem sobreposição alta: possível texto adicional; requer revisão de escopo
contos/macn010.txt Conto, Virginius, 1864 contos/macn007.txt 0.0 sem sobreposição alta: possível texto adicional; requer revisão de escopo
contos/macn011.txt CONTO, Casada e viúva, 1864 contos/macn005.txt 0.0012 sem sobreposição alta: possível texto adicional; requer revisão de escopo
contos/macn012.txt Conto, O anjo das donzelas, 1864 contos/macn007.txt 0.0 sem sobreposição alta: possível texto adicional; requer revisão de escopo
contos/macn013.txt Conto, Q

## 9. Cruzamento efetivo com a bibliografia da ABL

O quadro inicial registrava candidatos de arquivo. Aqui resolvemos cada referência contra os caminhos realmente extraídos, sem converter correspondência por categoria em confirmação de uma edição completa. O CSV final registra o(s) arquivo(s) que dão suporte ao mapeamento e mantém separadas as situações direta, distribuída/parcial, tradução excluída e lacuna.

In [13]:
def resolve_reference(reference):
    found = []
    if reference == "nenhum arquivo identificado":
        return found
    for piece in reference.split(";"):
        piece = piece.strip()
        if "*" in piece:
            found.extend(
                p.relative_to(SOURCE_ROOT).as_posix()
                for p in sorted(SOURCE_ROOT.glob(piece))
                if p.is_file()
            )
        else:
            path = SOURCE_ROOT / piece
            if path.is_file():
                found.append(piece)
    return sorted(set(found))

abl_audit = []
for title, reference, initial_status in abl_crosswalk:
    files = resolve_reference(reference)
    if initial_status.startswith("excluído"):
        final_status = "tradução identificada e excluída do corpus autoral"
    elif initial_status == "lacuna":
        final_status = "não localizado no índice-fonte"
    elif initial_status.startswith("localizado"):
        final_status = "correspondência direta no pacote"
    elif files:
        final_status = "material relacionado localizado; equivalência integral não verificada"
    else:
        final_status = "sem arquivo correspondente localizado"
    abl_audit.append({
        "obra_ABL": title,
        "referencia_candidata": reference,
        "arquivos_existentes": " | ".join(files),
        "status_revisado": final_status,
        "nota_inicial": initial_status,
    })

abl_audit_path = DATA_DIR / "cruzamento_bibliografia_abl.csv"
with abl_audit_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(abl_audit[0].keys()))
    writer.writeheader()
    writer.writerows(abl_audit)

abl_counts = Counter(x["status_revisado"] for x in abl_audit)
print("Títulos da lista ABL conferidos:", len(abl_audit))
print("Resumo:", dict(abl_counts))
for row in abl_audit:
    if row["status_revisado"] != "correspondência direta no pacote":
        print(row["obra_ABL"], "=>", row["status_revisado"], "|", row["arquivos_existentes"] or "—")

Títulos da lista ABL conferidos: 32
Resumo: {'tradução identificada e excluída do corpus autoral': 1, 'correspondência direta no pacote': 24, 'material relacionado localizado; equivalência integral não verificada': 6, 'não localizado no índice-fonte': 1}
Queda que as mulheres têm para os tolos => tradução identificada e excluída do corpus autoral | miscelanea/mams02.txt
Teatro => material relacionado localizado; equivalência integral não verificada | teatro/matt01.txt | teatro/matt02.txt | teatro/matt03.txt | teatro/matt04.txt | teatro/matt05.txt | teatro/matt06.txt | teatro/matt07.txt | teatro/matt08.txt | teatro/matt09.txt | teatro/matt10.txt
Poesias completas => material relacionado localizado; equivalência integral não verificada | poesia/maps01.txt | poesia/maps02.txt | poesia/maps03.txt | poesia/maps04.txt | poesia/maps05.txt | poesia/maps06.txt | poesia/maps07.txt
Crítica (1910) => material relacionado localizado; equivalência integral não verificada | critica/mact01.txt | criti

## 10. Decisão após a auditoria e atualização do manifesto

Contos avulsos contidos ou com sobreposição alta não serão concatenados de novo ao mestre. Os sem sobreposição alta são acrescentados como peças autorais distintas, sem afirmar que sejam inéditas ou que nunca tenham sido reunidas em outras coletâneas. Casos parcialmente comparáveis ou sem cabeçalho reconhecido ficam pendentes.

O cruzamento ABL agora resolve presença de arquivo de modo verificável. “Correspondência direta” quer dizer que o caminho indicado existe; não certifica edição, completude da coleção ou fidelidade a exemplar de primeira edição.

In [14]:
# Recalcula a versão final incluindo os contos avulsos sem sobreposição alta.
story_by_path = {r["source_file"]: r for r in story_audit}
inventory_by_path = {r["source_file"]: r for r in inventory}
for row in audit:
    if row["role"] == "conto_avulso":
        result = story_by_path[row["source_file"]]
        selected = result["decision"].startswith("sem sobreposição alta:")
        row["include_in_master"] = selected
        row["decision"] = ("incluído como texto avulso sem sobreposição alta"
                           if selected else "pendente: sobreposição ou extração requer revisão")
        inventory_by_path[row["source_file"]]["include_in_master"] = selected
        inventory_by_path[row["source_file"]]["decision"] = row["decision"]
        result["selected_for_master"] = selected

included = [row for row in audit if row["include_in_master"]]
parts = []
for row in included:
    source_text = (SOURCE_ROOT / row["source_file"]).read_text(encoding="utf-8")
    label = row.get("title_in_contents") or row["source_file"]
    parts.append(
        f"\n\n===== INÍCIO DO ITEM: {label} | categoria={row['category']} | fonte={row['source_file']} =====\n\n"
        + source_text
        + f"\n\n===== FIM DO ITEM: {row['source_file']} =====\n"
    )
master_text = "".join(parts)
MASTER_PATH.write_text(master_text, encoding="utf-8")
master_hash = hashlib.sha256(MASTER_PATH.read_bytes()).hexdigest()

for outpath, rows in ((inventory_path, inventory), (audit_path, audit), (story_audit_path, story_audit)):
    with outpath.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

supplemental_count = sum(r.get("selected_for_master", False) for r in story_audit)
print("Contos avulsos incluídos após comparação:", supplemental_count)
print("Casos mantidos pendentes:", len(story_audit) - supplemental_count)
print("Total de itens no corpus mestre final:", len(included))
print("Caracteres do mestre final:", len(master_text))
print("Bytes UTF-8 do mestre final:", MASTER_PATH.stat().st_size)
print("SHA-256 do mestre final:", master_hash)

manifest.update({
    "unindexed_short_story_audit": {
        "method": "corpo após nota 'Publicado originalmente'; chave alfanumérica sem diacríticos; cobertura de n-gramas distintos de 32 caracteres contra 7 coletâneas",
        "ngrams_per_story_threshold_high": 0.85,
        "ngrams_per_story_threshold_partial": 0.40,
        "report_file": story_audit_path.name,
        "counts": dict(story_counts),
        "automatic_addition_to_master": False,
    },
    "abl_bibliography_crosswalk": {
        "reference": ABL_BIBLIOGRAPHY,
        "titles_checked": len(abl_audit),
        "report_file": abl_audit_path.name,
        "counts": dict(abl_counts),
        "direct_match_means": "mapped source path exists; does not establish edition equivalence or exhaustive coverage",
    },
    "master_is_working_version": True,
    "full_bibliographic_completeness_claimed": False,
})
manifest.update({
    "included_authorial_indexed_items": sum(r["in_index_116"] and r["include_in_master"] for r in audit),
    "final_master_item_count": len(included),
    "supplemental_avulso_items_included": supplemental_count,
    "unindexed_short_story_files_preserved_for_review": len(story_audit) - supplemental_count,
    "master_file": MASTER_PATH.name,
    "master_sha256": master_hash,
    "master_utf8_bytes": MASTER_PATH.stat().st_size,
    "source_text_encoding": "Windows-1252 (CP1252), verificado nos membros do ZIP",
    "extracted_text_encoding": "UTF-8",
    "limitations": [
        "Itens não equivalem a livros.",
        "A bibliografia completa não foi comprovada; ver cruzamento ABL.",
        "Contos avulsos entraram por baixa sobreposição textual com as sete coletâneas, não por prova de ineditismo.",
        "Transcrições citam edições modernas; preservar e revisar a proveniência editorial.",
        "O corpus mestre mantém cabeçalhos e notas editoriais da transcrição.",
    ],
})
# Verificação de ida: cada cópia UTF-8 equivale à decodificação CP1252 do membro original.
with ZipFile(ARCHIVE_PATH) as original_zip:
    for row in inventory:
        if row["source_file"].lower().endswith(".txt"):
            expected = original_zip.read("machado/" + row["source_file"]).decode("cp1252")
            actual = (SOURCE_ROOT / row["source_file"]).read_text(encoding="utf-8")
            assert actual == expected, f"Divergência na conversão: {row['source_file']}"
all_working_text = "\n".join((SOURCE_ROOT / r["source_file"]).read_text(encoding="utf-8") for r in inventory)
assert "\ufffd" not in all_working_text
assert len(included) == 112 + supplemental_count
assert supplemental_count == 130
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Conversão dos textos: CP1252 → UTF-8 validada contra todos os membros TXT do ZIP.")
print("U+FFFD nas cópias extraídas:", all_working_text.count("\ufffd"))
print("Manifesto atualizado:", manifest_path)
print("Hashes preservados:", manifest["archive_sha256_calculated"], manifest["master_sha256"])

Contos avulsos incluídos após comparação: 130
Casos mantidos pendentes: 0
Total de itens no corpus mestre final: 242
Caracteres do mestre final: 13740184
Bytes UTF-8 do mestre final: 14192790
SHA-256 do mestre final: f5562aa740556fae3435cfacf5b54c414909cd54f2fb627067132225791cc5b3


Conversão dos textos: CP1252 → UTF-8 validada contra todos os membros TXT do ZIP.
U+FFFD nas cópias extraídas: 0
Manifesto atualizado: /workspaces/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projetos/machado-assis/dados/manifesto_corpus.json
Hashes preservados: 772463b1553c1b0ff1fc0360768b31f59b488f7a52d44cc92c3e31ca289acce9 f5562aa740556fae3435cfacf5b54c414909cd54f2fb627067132225791cc5b3
